# Yoga Pose Detection — Colab-ready experiment

This repaired notebook trains the project CNN from the serialized landmark datasets committed in this repository. It runs from any Colab working directory and does not depend on missing local image folders.

> Educational computer-vision demo only. Predictions can be wrong and are not medical advice or a substitute for a qualified yoga instructor.

Dataset note: the original notebook expected Clean_dataset/TRAIN/<class> image directories, but GitHub contains prepared landmark pickles under Deep Learning Code/Clean_dataset/pickled_data. This version uses those checked-in artifacts directly.


In [ ]:
# Imports and reproducibility
import pickle
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)


In [ ]:
# Download the versioned landmark datasets from this repository.
RAW_BASE = "https://raw.githubusercontent.com/kartikverma23/Minor/main/Deep%20Learning%20Code/Clean_dataset/pickled_data"
DATA_DIR = Path("minor_data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_dataset(split):
    target = DATA_DIR / split
    if not target.exists():
        urllib.request.urlretrieve(f"{RAW_BASE}/{split}", target)
    with target.open("rb") as handle:
        return pickle.load(handle)

train_raw = fetch_dataset("TRAIN")
test_raw = fetch_dataset("TEST")
print("Dataset sizes:", {name: len(value["data"]) for name, value in {"TRAIN": train_raw, "TEST": test_raw}.items()})


In [ ]:
# Validate and convert the serialized data.
CLASS_NAMES = ["Down Dog", "Goddess", "Plank", "Tree", "Warrior II"]
X_train = np.asarray(train_raw["data"], dtype=np.float32)
y_train = np.asarray(train_raw["labels"], dtype=np.float32)
X_test = np.asarray(test_raw["data"], dtype=np.float32)
y_test = np.asarray(test_raw["labels"], dtype=np.float32)

assert X_train.shape[1:] == (33, 2) and X_test.shape[1:] == (33, 2)
assert y_train.shape[1] == len(CLASS_NAMES) and y_test.shape[1] == len(CLASS_NAMES)
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)
print("Train class counts:", dict(zip(CLASS_NAMES, np.bincount(np.argmax(y_train, axis=1), minlength=len(CLASS_NAMES)))))


In [ ]:
# Build efficient TensorFlow input pipelines.
BATCH_SIZE = 32
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train), seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_ds = (tf.data.Dataset.from_tensor_slices((X_test, y_test))
           .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))


In [ ]:
# Define the landmark classifier.
def create_model():
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(33, 2)),
        tf.keras.layers.Conv1D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.MaxPooling1D(),
        tf.keras.layers.Conv1D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="categorical_crossentropy",
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
    )
    return model

model = create_model()
model.summary()


In [ ]:
# Train with early stopping so the notebook is practical on a Colab CPU runtime.
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, mode="max", restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, verbose=1),
]
history = model.fit(train_ds, validation_data=test_ds, epochs=30, callbacks=callbacks, verbose=1)


In [ ]:
# Plot training history.
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[0], title="Accuracy")
history_df[["loss", "val_loss"]].plot(ax=axes[1], title="Loss")
plt.tight_layout()
plt.show()


In [ ]:
# Evaluate and print per-class metrics.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
probabilities = model.predict(X_test, verbose=0)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(probabilities, axis=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


In [ ]:
# Save the repaired model in the current Colab session.
MODEL_PATH = Path("pose_model.keras")
model.save(MODEL_PATH)
print("Saved:", MODEL_PATH.resolve(), "bytes:", MODEL_PATH.stat().st_size)


In [ ]:
# Run one deterministic prediction from the bundled test set.
sample_index = 0
sample_probabilities = model.predict(X_test[sample_index:sample_index + 1], verbose=0)[0]
predicted_index = int(np.argmax(sample_probabilities))
actual_index = int(np.argmax(y_test[sample_index]))
print("Predicted:", CLASS_NAMES[predicted_index], f"({sample_probabilities[predicted_index] * 100:.2f}% confidence)")
print("Actual:", CLASS_NAMES[actual_index])
print("Prediction check:", "PASS" if predicted_index == actual_index else "completed with a mismatch on this sample")


## Runtime result

If every code cell completes, the repaired Minor notebook has loaded the committed data, trained the CNN, evaluated it, saved pose_model.keras, and produced a sample prediction. The original Flask app remains a separate local/web-app path and still requires MediaPipe plus the bundled scikit-learn model.
